# Case Study — Module M5: Machine Learning Fundamentals
## Regression Algorithm Performance Metrics: AutoPredict Analytics

| | |
|---|---|
| **Author** | Jose Marcel Lopez Pino |
| **Framework** | CRISP-DM + LEAN |
| **Phases** | Phase 1–2–4–5: Business Understanding + Data Understanding + Modeling + Evaluation |
| **Module** | M5 — Machine Learning Fundamentals (Alkemy Bootcamp) |
| **Dataset** | AutoPredict S.A. — Used Vehicle Pricing (n=35: 4 original + 31 synthetic, random_state=42) |
| **Case** | L6 — Regression Algorithm Performance Metrics |
| **Date** | 2026-04 |

---

> **Executive Summary:**
> AutoPredict S.A. has developed a linear regression model to estimate used vehicle sale prices based on vehicle age, mileage, and number of doors. The original case dataset (n=4) was extended to n=35 records — 4 original + 31 synthetic generated under the same business rules — to enable statistically meaningful metric evaluation. This decision is explicitly documented in the Decisions Log (D1). The model achieves R²=0.9541 and MAPE=6.74% on the test set, demonstrating strong predictive fit. Five performance metrics (MAE, MSE, RMSE, R², MAPE) are computed, interpreted in business terms, and accompanied by a comparative visualization of actual vs. predicted prices and a prioritized improvement roadmap.

## Table of Contents

1. [Phase 1 — Business Understanding](#phase-1)
2. [Phase 2 — Data Understanding](#phase-2)
3. [Phase 4 — Modeling](#phase-4)
4. [Phase 5 — Evaluation and Reflection](#phase-5)
5. [LEAN Filter — Waste Elimination Review](#lean-filter)
6. [Decisions Log](#decisions-log)
7. [Next Steps](#next-steps)

---

## 1. Phase 1 — Business Understanding <a id='phase-1'></a>

### Problem Statement Canvas

| Element | Description |
|---|---|
| **Business Problem** | AutoPredict S.A. has no documented procedure to evaluate whether its linear regression model for used vehicle pricing is fit for production deployment. |
| **Business Impact** | An unvalidated model exposes dealership clients to systematic pricing errors — overpricing reduces sales volume; underpricing erodes margin. Both outcomes damage client trust and AutoPredict's commercial reputation. |
| **Decision to Support** | Should the current linear regression model be deployed to production, or does it require further refinement before client delivery? |
| **Analytical Question** | What are the MAE, MSE, RMSE, R², and MAPE of the model on a representative dataset, and do these metrics meet the precision and reliability criteria required for deployment? |
| **Success Metrics** | R² > 0.80 on test set; MAPE < 15%; all five required metrics computed and interpreted in business terms. |
| **Proposed Approach** | Extend original n=4 dataset to n=35 (4 original + 31 synthetic under same business rules); train multiple linear regression (scikit-learn) on 80/20 split; compute all metrics; visualize actual vs. predicted prices; document improvement decisions. |

### Personal Perspective — ICI Background

From an operations engineering standpoint, deploying an unvalidated model is equivalent to releasing a product to market without quality inspection. In manufacturing, this is called **escaping defects** — the most expensive failure mode because the cost is borne by the customer, not the plant. The regression metrics in this case serve the same function as a **process capability index (Cpk)**: they quantify whether the process (model) is capable of meeting the specification (pricing accuracy) before the product (prediction) reaches the client. LEAN principle: **inspect at the gate, not at the customer**.

---

## 2. Phase 2 — Data Understanding <a id='phase-2'></a>

In [ ]:
# =============================================================================
# IMPORTS
# Standard Library
import warnings

# Core Data Science
import numpy as np
import pandas as pd

# ML — Regression + Model Selection + Metrics
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Configuration
warnings.formatwarning = lambda msg, *args, **kwargs: f'Warning: {msg}\n'
warnings.simplefilter('always')

pd.set_option('display.float_format', '{:.4f}'.format)
pd.set_option('display.max_columns', None)

print('Environment ready.')

In [ ]:
# =============================================================================
# DATASET — 4 original records + 31 synthetic under the same business rules
# Business rules:
#   - Price decreases with Age and Mileage (depreciation logic)
#   - Mileage correlates with Age (~10,000 km/year +/- noise)
#   - Doors: 2 or 4 (categorical, minor positive price effect)
#   - Price range: $7,000-$20,000 (used vehicle market)
#   - random_state=42 for reproducibility
# =============================================================================

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Original 4 records from the case consigna
original = pd.DataFrame({
    'Age_years':  [5,     3,     7,     2    ],
    'Mileage_km': [50000, 30000, 70000, 25000],
    'Doors':      [4,     2,     4,     2    ],
    'Price_USD':  [12000, 15000, 9000,  16000],
    'Source':     ['original'] * 4
})

# 31 synthetic records
N_SYNTH  = 31
age      = np.random.randint(1, 11, N_SYNTH)
mileage  = age * 10000 + np.random.randint(-4000, 4000, N_SYNTH)
mileage  = np.clip(mileage, 5000, 100000)
doors    = np.random.choice([2, 4], N_SYNTH)
noise    = np.random.randint(-800, 800, N_SYNTH)
price    = (20000 - age * 1000 - mileage * 0.05 + doors * 200 + noise).astype(int)
price    = np.clip(price, 7000, 20000)

synthetic = pd.DataFrame({
    'Age_years':  age,
    'Mileage_km': mileage,
    'Doors':      doors,
    'Price_USD':  price,
    'Source':     ['synthetic'] * N_SYNTH
})

# Combine and assign IDs
df = pd.concat([original, synthetic], ignore_index=True)
df.insert(0, 'ID', range(1, len(df) + 1))

print('=== Full Dataset (n=35) ===')
print(df.to_string(index=False))
print()
print(f'Shape     : {df.shape}')
print(f'Original  : 4 records | Synthetic: {N_SYNTH} records')

In [ ]:
# =============================================================================
# DATA QUALITY CHECK
# =============================================================================

print('=== Data Types ===')
print(df.dtypes)
print()
print('=== Missing Values ===')
print(df.isnull().sum())
print()
print('=== Descriptive Statistics ===')
print(df[['Age_years', 'Mileage_km', 'Doors', 'Price_USD']].describe().T)

In [ ]:
# =============================================================================
# CORRELATION MATRIX — feature-target relationships
# =============================================================================

print('=== Correlation Matrix ===')
print(df[['Age_years', 'Mileage_km', 'Doors', 'Price_USD']].corr().round(4))

**Data Understanding — Key Observations:**

- Dataset contains 35 records: 4 original (from case consigna) + 31 synthetic generated under the same business rules.
- No missing values — dataset is complete and ready for modeling.
- All variables are numeric — no encoding required.
- Strong negative correlation between Price_USD and both Age_years and Mileage_km — consistent with vehicle depreciation logic.
- Age_years and Mileage_km are highly correlated (r≈0.996) — reflects real-world usage (~10,000 km/year). Documented as multicollinearity risk in D1.
- Doors shows a weak negative correlation with Price_USD — counterintuitive but consistent with the synthetic generation noise at this sample size.

---

## 3. Phase 4 — Modeling <a id='phase-4'></a>

In [ ]:
# =============================================================================
# FEATURE MATRIX AND TARGET VECTOR
# =============================================================================

FEATURES = ['Age_years', 'Mileage_km', 'Doors']
TARGET   = 'Price_USD'

X = df[FEATURES]
y = df[TARGET]

print(f'Features : {FEATURES}')
print(f'Target   : {TARGET}')
print(f'X shape  : {X.shape}')
print(f'y shape  : {y.shape}')

In [ ]:
# =============================================================================
# TRAIN-TEST SPLIT — 80/20
# random_state=42 for reproducibility (non-negotiable in ML portfolios)
# =============================================================================

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size    = 0.2,
    random_state = RANDOM_STATE
)

print(f'Train set : {X_train.shape[0]} records ({X_train.shape[0] / len(df) * 100:.0f}%)')
print(f'Test set  : {X_test.shape[0]} records ({X_test.shape[0] / len(df) * 100:.0f}%)')

In [ ]:
# =============================================================================
# MODEL TRAINING — Multiple Linear Regression
# =============================================================================

model = LinearRegression()
model.fit(X_train, y_train)

print('=== Model Coefficients ===')
for feature, coef in zip(FEATURES, model.coef_):
    print(f'  {feature:<15}: {coef:>10.4f}')
print(f'  {"Intercept":<15}: {model.intercept_:>10.4f}')

In [ ]:
# =============================================================================
# PREDICTIONS
# =============================================================================

y_pred = model.predict(X_test)

comparison = pd.DataFrame({
    'Vehicle_ID':     X_test.index + 1,
    'Actual_USD':     y_test.values,
    'Predicted_USD':  y_pred.round(2),
    'Error_USD':      (y_test.values - y_pred).round(2),
    'Error_pct':      ((y_test.values - y_pred) / y_test.values * 100).round(2)
})

print('=== Actual vs. Predicted Prices ===')
print(comparison.to_string(index=False))

---

## 4. Phase 5 — Evaluation and Reflection <a id='phase-5'></a>

In [ ]:
# =============================================================================
# PERFORMANCE METRICS — MAE, MSE, RMSE, R², MAPE
# =============================================================================

mae  = mean_absolute_error(y_test, y_pred)
mse  = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2   = r2_score(y_test, y_pred)
mape = np.mean(np.abs((y_test.values - y_pred) / y_test.values)) * 100

metrics = pd.DataFrame({
    'Metric':         ['MAE', 'MSE', 'RMSE', 'R²', 'MAPE (%)'],
    'Value':          [mae, mse, rmse, r2, mape],
    'Unit':           ['USD', 'USD²', 'USD', 'dimensionless', '%'],
    'Interpretation': [
        'Average absolute prediction error in USD',
        'Penalizes large errors more heavily than MAE',
        'Typical prediction error in the same unit as target',
        'Proportion of price variance explained by the model',
        'Average prediction error as % of actual price — business-friendly'
    ]
})

print('=== Model Performance Metrics ===')
print(metrics.to_string(index=False))

In [ ]:
# =============================================================================
# VISUALIZATION — Actual vs. Predicted Prices
# =============================================================================

from pathlib import Path

fig, ax = plt.subplots(figsize=(9, 5))

x_pos = np.arange(len(y_test))
width = 0.35

ax.bar(x_pos - width / 2, y_test.values, width,
       label='Actual Price',    color='#2E86AB', alpha=0.85)
ax.bar(x_pos + width / 2, y_pred,         width,
       label='Predicted Price', color='#E84855', alpha=0.85)

ax.set_xlabel('Vehicle (test set)',  fontsize=11)
ax.set_ylabel('Price (USD)',          fontsize=11)
ax.set_title(
    f'AutoPredict S.A. — Actual vs. Predicted Used Vehicle Prices\n'
    f'MAE: ${mae:,.0f} | RMSE: ${rmse:,.0f} | R²: {r2:.4f} | MAPE: {mape:.2f}%',
    fontsize=11
)
ax.set_xticks(x_pos)
ax.set_xticklabels([f'V{int(vid)}' for vid in comparison['Vehicle_ID'].values])
ax.legend(fontsize=10)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))

plt.tight_layout()

output_dir = Path('../reports')
output_dir.mkdir(parents=True, exist_ok=True)
fig_path   = output_dir / 'actual_vs_predicted_prices.png'
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.show()

print(f'Figure saved.' if fig_path.exists() else 'Save failed.')

### Reflection Q&A

**Q1: How precise is the model?**

With R²=0.9541 the model explains 95.4% of the price variance in the test set — strong predictive fit for a three-feature linear model. MAPE=6.74% means predictions are within ~7% of the actual vehicle price on average, well below the 15% deployment threshold established for dealership pricing. RMSE=$790.85 quantifies the typical absolute error in USD — acceptable for a first-version model at this price range ($7,000–$20,000). The one outlier in the test set (Vehicle 27: error 21.99%) is a boundary artifact caused by the $7,000 price floor in the synthetic generation — documented in D5.

**Q2: What decisions would you make to improve model performance?**

| Priority | Decision | Rationale |
|---|---|---|
| 🔴 Critical | Collect real transaction data (n ≥ 500) from client dealerships | Synthetic data captures depreciation logic but not market-specific pricing dynamics |
| 🔴 Critical | Add features: brand, fuel type, transmission, vehicle condition | Age and mileage alone do not capture full price drivers in used vehicle markets |
| 🟡 Important | Evaluate Ridge or Lasso regularization | Age_years and Mileage_km are highly correlated (r=0.996) — multicollinearity risk for OLS |
| 🟡 Important | Apply k-fold cross-validation (k=5) | More stable performance estimates than a single 80/20 split |
| 🟢 Enhancement | Test Gradient Boosting (XGBoost) | Captures non-linear mileage depreciation effects (accelerates past 100k km) |

---

## 5. LEAN Filter — Waste Elimination Review <a id='lean-filter'></a>

| LEAN Waste Type | Identified? | Detail | Eliminated? |
|---|---|---|---|
| **Defects** | ✅ Yes | Original n=4 produces degenerate R² (1 test record) — statistically meaningless | ✅ Extended to n=35 (4 original + 31 synthetic) under same business rules (D1) |
| **Overprocessing** | ⚠️ Partial | Five metrics computed — MAPE is beyond consigna requirements | ✅ Justified — MAPE is most business-interpretable metric for dealership pricing |
| **Waiting** | ❌ No | Fast execution — no computation bottleneck | N/A |
| **Inventory (data)** | ✅ Yes | Dataset generated inline — no external file required | ✅ No unnecessary data movement |
| **Motion** | ❌ No | Single notebook, linear execution | N/A |
| **Transport** | ❌ No | No unnecessary data movement between systems | N/A |

---

## 6. Decisions Log <a id='decisions-log'></a>

| # | Decision | Rationale | LEAN Value | Alternative Considered |
|---|---|---|---|---|
| D1 | Extended dataset from n=4 to n=35 (4 original + 31 synthetic) | n=4 produces degenerate R² with 1 test record — statistically meaningless for model evaluation. Synthetic records preserve case business rules: depreciation logic, price range $7k–$20k, mileage~age correlation | Eliminates analytical defect at source — simulation-first: mistakes in simulation cost time, not money | Keep n=4 and document limitation only (weaker portfolio signal) |
| D2 | random_state=42 fixed for data generation and train-test split | Reproducibility is non-negotiable in ML portfolios — any reviewer must reproduce identical results | Eliminates non-determinism waste | No fixed seed (irreproducible — unacceptable) |
| D3 | MAPE included beyond consigna requirements | Most business-friendly regression metric for vehicle pricing — expresses error as % of actual price, directly comparable to dealership pricing tolerance margins | Adds client communication value at zero extra cost | Omit MAPE (meets minimum consigna but loses business interpretability) |
| D4 | Improvement decisions framed as priority table (🔴🟡🟢) | Business stakeholders need prioritized action plans, not unordered option lists | Converts analysis into actionable recommendations | Plain list (analytically complete but not decision-ready) |
| D5 | Vehicle 27 outlier (error 21.99%) documented rather than removed | Artifact of $7,000 price floor in synthetic generation — removing it would hide a known data quality issue | Transparency over cosmetic metric improvement | Remove outlier (inflates R² artificially) |

---

## 7. Next Steps <a id='next-steps'></a>

| Priority | Next Step | Scope |
|---|---|---|
| 🔴 Immediate | Complete `reports/executive_summary.md` using real metrics and chart from this notebook | M5 — CBL deliverable |
| 🟡 Short-term | Apply Ridge/Lasso regression on a larger real used vehicle dataset and compare metrics against this baseline | M5–M6 |
| 🟢 Long-term | Integrate this regression evaluation pipeline as a reusable `src/evaluation.py` module for future ML projects | Portfolio |

---

*Framework: CRISP-DM + LEAN | Methodology: Case-Based Learning (CBL)*

**Jose Marcel Lopez Pino**  
Data Scientist — Operations, Analytics & Process Optimization  
Bootcamp: Fundamentos de Ciencia de Datos — SENCE/Alkemy (2025–2026)

[![GitHub](https://img.shields.io/badge/GitHub-joselopezp-181717?style=flat&logo=github)](https://github.com/joselopezp)
[![LinkedIn](https://img.shields.io/badge/LinkedIn-jose--lopez--pino-0077B5?style=flat&logo=linkedin)](https://www.linkedin.com/in/jose-lopez-pino/)